# 04_verifier — 2-Hop Fast E2E
Uses existing 01 reranker from `roberta_e2e_v2`, trains 2-Hop cohorts only, supports 100-claim preview then full Dev.


In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ['PYTHONUNBUFFERED']='1'
import base64,io,zipfile
BUNDLE_B64='...'
if Path('/content').is_dir():
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as archive: archive.extractall('/content')
    os.environ['HOVER_E2E_ROOT']='/content/e2e_v2'
    print('[bootstrap] latest e2e_v2 code synchronized to /content/e2e_v2',flush=True)
DRIVE_CANDIDATE=Path('/content/drive/MyDrive/hover-project')
DRIVE_PROJECT=DRIVE_CANDIDATE if DRIVE_CANDIDATE.is_dir() else None
if DRIVE_PROJECT is None and Path('/content').is_dir():
    try:
        from google.colab import drive as _fast2_drive
        _fast2_drive.mount('/content/drive')
        DRIVE_PROJECT=DRIVE_CANDIDATE if DRIVE_CANDIDATE.is_dir() else None
    except ImportError:
        pass
roots=([Path(os.environ['HOVER_E2E_ROOT']).expanduser()] if os.environ.get('HOVER_E2E_ROOT') else [])+[Path.cwd(),*Path.cwd().parents,Path('/content/e2e_v2'),Path('/content/고준서/e2e_v2'),Path('/content/Fact-Verfication')]+([DRIVE_PROJECT] if DRIVE_PROJECT else [])
candidates=[candidate for root in roots for candidate in (root,root/'e2e_v2',root/'고준서/e2e_v2')]
E2E_ROOT=next((p.resolve() for p in candidates if (p/'pyproject.toml').is_file() and (p/'src/hover_e2e').is_dir()),None)
if E2E_ROOT is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_PROJECT=Path('/content/drive/MyDrive/hover-project')
        mounted_candidates=[DRIVE_PROJECT,DRIVE_PROJECT/'e2e_v2',DRIVE_PROJECT/'고준서/e2e_v2']
        E2E_ROOT=next((p.resolve() for p in mounted_candidates if (p/'pyproject.toml').is_file() and (p/'src/hover_e2e').is_dir()),None)
    except ImportError:
        pass
if E2E_ROOT is None: raise FileNotFoundError('현재 커널과 마운트된 Drive에서 e2e_v2 소스를 찾지 못했습니다. /content/e2e_v2 또는 MyDrive/hover-project/고준서/e2e_v2에 폴더가 필요합니다.')
subprocess.check_call([sys.executable,'-m','pip','install','-q','-e',f'{E2E_ROOT}[gpu,test]'])
sys.path.insert(0,str(E2E_ROOT/'src')) if str(E2E_ROOT/'src') not in sys.path else None
from hover_e2e.colab_runtime import prepare_runtime
RUN_NAME=os.environ.get('HOVER_FAST2_RUN_NAME','roberta_e2e_v2_2hop_fast')
HOPS='2'  # Fixed scope; original 01 reranker is reused
CTX=prepare_runtime(E2E_ROOT,drive_project=DRIVE_PROJECT,run_name=RUN_NAME)
CONFIG=E2E_ROOT/'configs/fast2.json'
import torch
assert torch.cuda.is_available(),'GPU 런타임으로 변경하세요.'
print(torch.cuda.get_device_name(0),'| hops',HOPS)

# FAST2: 0 = full 2-Hop Dev (1,126); 100 = 100-claim preview.
DEV_LIMIT = int(os.environ.get('HOVER_FAST_DEV_LIMIT', '100'))
print(f'[fast2] run={RUN_NAME}; retrieval source=roberta_e2e_v2; dev_limit={DEV_LIMIT}; config={CONFIG}',flush=True)


In [ ]:
subprocess.run([sys.executable,'-u','-m','hover_e2e.cli','verifier','--config',str(CONFIG),'--data-root',str(CTX.data_root),'--output-root',str(CTX.output_root),'--run-name',RUN_NAME,'--hops',HOPS,'--dev-limit',str(DEV_LIMIT)],check=True)
print('Verifier complete. 다음: 05_e2e.ipynb')

print('[fast2] 04_verifier done; remember same HOVER_FAST2_RUN_NAME for next notebook')
